# Week 6 · Day 3 — `JOIN`: stitch tables together

*Your data lives in several tables. A join is how you match them up — like matching a client list to a matter list.*

**By the end you'll have shipped:** a **profit-by-store report** that `JOIN`s `coffee_orders` to a `menu` (for unit costs) and to `stores` (for regions) — `INNER` vs `LEFT` joins, run through the same `run_sql` helper.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1b · SQL foundations → M4 · Snowflake (Week 6) |
| **Prerequisites** | W6D2 (`GROUP BY`), Week 2 Day 3 (`merge`) |
| **Est. time** | ~30 min |
| **Capstone slice** | *Query* related legal tables — matters matched to clients / attorneys |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — Snowflake if credentials exist, else DuckDB |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain **why** data is split across tables and what a **key** is.
- Write an **`INNER JOIN ... ON`** to combine two tables on a shared column.
- Use **table aliases** (`o`, `m`) and qualify columns (`o.price`) to avoid ambiguity.
- Use a **`LEFT JOIN`** to keep every row from the first table, even with no match.
- **Join then `GROUP BY`** to compute a grouped metric across tables (profit by region).

### ⚖️ Why it matters

The firm won't keep everything in one giant table. **Matters** live in one, **clients** in another, **attorneys** in a third — each fact stored once, in its natural home. A **`JOIN`** re-connects them on a shared key: *"match every matter to its client's industry."* It's the SQL twin of pandas `merge`, and the everyday legal act of **matching a client list to a matter list**.

### ⚙️ Setup

This lesson loads **three** tables: `coffee_orders`, `menu` (item → unit cost), and `stores` (store → region). Same `run_sql` helper.

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'coffee_orders': [{'order_id': 'O-5001', 'date': '2026-03-07', 'item': 'Cappuccino', 'size': 'S', 'category': 'Espresso Drink', 'price': 3.75, 'payment': 'Cash', 'store': 'Downtown'}, {'order_id': 'O-5002', 'date': '2026-03-07', 'item': 'Mocha', 'size': 'S', 'category': 'Espresso Drink', 'price': 4.5, 'payment': 'Card', 'store': 'Airport'}, {'order_id': 'O-5003', 'date': '2026-03-05', 'item': 'Cappuccino', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.25, 'payment': 'Card', 'store': 'Downtown'}, {'order_id': 'O-5004', 'date': '2026-03-05', 'item': 'Croissant', 'size': 'M', 'category': 'Food', 'price': 3.25, 'payment': 'App', 'store': 'Uptown'}, {'order_id': 'O-5005', 'date': '2026-03-06', 'item': 'Latte', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.5, 'payment': 'App', 'store': 'Downtown'}],
    'menu': [{'item': 'Latte', 'category': 'Espresso Drink', 'unit_cost': 1.2}, {'item': 'Cappuccino', 'category': 'Espresso Drink', 'unit_cost': 1.1}, {'item': 'Espresso', 'category': 'Espresso Drink', 'unit_cost': 0.8}, {'item': 'Mocha', 'category': 'Espresso Drink', 'unit_cost': 1.4}, {'item': 'Croissant', 'category': 'Food', 'unit_cost': 0.9}, {'item': 'Cold Brew', 'category': 'Cold', 'unit_cost': 1.15}],
    'stores': [{'store': 'Downtown', 'city': 'Rivertown', 'region': 'Central'}, {'store': 'Uptown', 'city': 'Rivertown', 'region': 'North'}, {'store': 'Airport', 'city': 'Rivertown', 'region': 'South'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

In [ ]:
# Peek at the three tables and their shared keys.
print("coffee_orders (has `item` and `store`):")
display(run_sql("SELECT order_id, item, price, store FROM coffee_orders LIMIT 3"))
print("menu (has `item` and `unit_cost`):")
display(run_sql("SELECT * FROM menu LIMIT 3"))
print("stores (has `store` and `region`):")
display(run_sql("SELECT * FROM stores"))

**Notice the shared keys:** `coffee_orders.item` matches `menu.item`; `coffee_orders.store` matches `stores.store`. Those shared columns are what a join lines up on.

### 1 · `INNER JOIN` — combine two tables on a key  →  like `pd.merge(...)`

An order knows its `item` but not that item's **cost** — that's in `menu`. `JOIN menu ON coffee_orders.item = menu.item` glues each order to its menu row, so we can compute **profit = price − unit_cost**.

In [ ]:
run_sql("""
    SELECT coffee_orders.order_id,
           coffee_orders.item,
           coffee_orders.price,
           menu.unit_cost,
           coffee_orders.price - menu.unit_cost AS profit
    FROM coffee_orders
    JOIN menu ON coffee_orders.item = menu.item
    ORDER BY profit DESC
    LIMIT 5
""")

**What just happened:** each order was matched to its menu row on `item`, giving us `unit_cost` alongside `price` so we could compute `profit`. Plain `JOIN` means `INNER JOIN`: only rows that match in **both** tables come through.

### 2 · Table aliases — shorter, clearer

Writing `coffee_orders.` everywhere is tedious. Give each table a short **alias** right after its name (`coffee_orders AS o`, or just `coffee_orders o`) and use it as a prefix. Prefixes also resolve ambiguity: both tables have an `item` column, so `o.item` says *which one*.

In [ ]:
run_sql("""
    SELECT o.order_id, o.item, o.price, m.unit_cost,
           o.price - m.unit_cost AS profit
    FROM coffee_orders AS o
    JOIN menu AS m ON o.item = m.item
    ORDER BY profit DESC
    LIMIT 5
""")

### 3 · `INNER` vs `LEFT` JOIN — what happens to non-matches?

- **`INNER JOIN`** keeps only rows that match in *both* tables. If an order's item isn't on the menu, that order **disappears**.
- **`LEFT JOIN`** keeps **every** row from the left (first) table; where there's no match, the right table's columns come back **NULL**.

When you don't want to silently drop rows, reach for `LEFT JOIN`. Let's prove the difference by adding an order for an item that's *not* in the menu.

In [ ]:
# add a "Smoothie" order — there's no Smoothie row in `menu`
run_sql("INSERT INTO coffee_orders VALUES ('O-9999','2026-03-09','Smoothie','L','Cold',6.50,'Card','Airport')")

print("INNER JOIN — the Smoothie order is DROPPED (no menu match):")
display(run_sql("""
    SELECT o.order_id, o.item, m.unit_cost
    FROM coffee_orders o
    JOIN menu m ON o.item = m.item
    WHERE o.order_id IN ('O-9999','O-5001')
"""))

print("LEFT JOIN — the Smoothie order is KEPT, with unit_cost = NULL:")
display(run_sql("""
    SELECT o.order_id, o.item, m.unit_cost
    FROM coffee_orders o
    LEFT JOIN menu m ON o.item = m.item
    WHERE o.order_id IN ('O-9999','O-5001')
"""))

**What just happened:** the `INNER JOIN` dropped the Smoothie order because no menu row matched; the `LEFT JOIN` kept it with `unit_cost = NULL` (shown as `None`/`NaN`). **This is the difference that bites people:** an inner join can silently shrink your data.

### 4 · Join, then `GROUP BY` — a cross-table report

Joins and aggregates combine naturally: join `coffee_orders` to `stores` to get each order's **region**, then `GROUP BY region` to total revenue. This is the real shape of warehouse reporting.

In [ ]:
run_sql("""
    SELECT s.region,
           COUNT(*)               AS num_orders,
           ROUND(SUM(o.price), 2) AS revenue
    FROM coffee_orders o
    JOIN stores s ON o.store = s.store
    GROUP BY s.region
    ORDER BY revenue DESC
""")

> **`Go Deeper 🔧` — a three-table join, then aggregate.** Chain joins to bring in *both* cost and region, then group to get **profit by region**. Each `JOIN ... ON` adds one more table.

In [ ]:
run_sql("""
    SELECT s.region,
           ROUND(SUM(o.price), 2)               AS revenue,
           ROUND(SUM(o.price - m.unit_cost), 2) AS profit
    FROM coffee_orders o
    JOIN menu   m ON o.item  = m.item
    JOIN stores s ON o.store = s.store
    GROUP BY s.region
    ORDER BY profit DESC
""")

> **`Common pitfalls ⚠️`**
>
> - **Always give a join a condition** (`ON o.item = m.item`). Forget it and you get a *cross join* — every row paired with every other row.
> - **Qualify shared column names** with a table alias (`o.item`, not bare `item`) or SQL can't tell which you mean.
> - **`INNER` silently drops non-matches.** If a report's totals look low, an inner join may be hiding rows — try `LEFT JOIN` to check.

### ✍️ Your turn

In [ ]:
# TODO 1: INNER JOIN coffee_orders to menu; show order_id, item, price, unit_cost

# TODO 2: add a `profit` column (price - unit_cost) and sort by it, highest first

# TODO 3: JOIN coffee_orders to stores, then GROUP BY city to total revenue per city


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    SELECT o.order_id, o.item, o.price, m.unit_cost
    FROM coffee_orders o
    JOIN menu m ON o.item = m.item
""")

# 2
run_sql("""
    SELECT o.order_id, o.item, o.price - m.unit_cost AS profit
    FROM coffee_orders o
    JOIN menu m ON o.item = m.item
    ORDER BY profit DESC
""")

# 3
run_sql("""
    SELECT s.city, ROUND(SUM(o.price), 2) AS revenue
    FROM coffee_orders o
    JOIN stores s ON o.store = s.store
    GROUP BY s.city
    ORDER BY revenue DESC
""")
```
</details>

### 🚀 Build the artifact — a profit-by-store report

The report a shop owner runs weekly: revenue **and** true profit per store, only possible by joining orders to their menu costs, then grouping.

In [ ]:
def profit_by_store() -> pd.DataFrame:
    """Revenue and profit per store — orders joined to menu costs."""
    return run_sql("""
        SELECT o.store,
               COUNT(*)                             AS num_orders,
               ROUND(SUM(o.price), 2)               AS revenue,
               ROUND(SUM(o.price - m.unit_cost), 2) AS profit,
               ROUND(SUM(o.price - m.unit_cost) / SUM(o.price) * 100, 1) AS margin_pct
        FROM coffee_orders o
        JOIN menu m ON o.item = m.item
        GROUP BY o.store
        ORDER BY profit DESC
    """)

report = profit_by_store()
print(f"Total profit across stores: ${report['profit'].sum():.2f}")
report

> **🔗 Your world — from coffee to matters.** A join is *matching clients to matters*. Picture a `clients` table (`client`, `industry`, `region`) beside your `matters` table:
>
> ```sql
> SELECT c.industry,
>        COUNT(*)                     AS num_matters,
>        ROUND(SUM(m.amount_billed), 2) AS total_billed
> FROM matters m
> JOIN clients c ON m.client = c.client     -- match each matter to its client
> WHERE m.status = 'Active'
> GROUP BY c.industry
> ORDER BY total_billed DESC;
> ```
>
> *Active billings by client industry* — a number no single table could answer alone. That's why joins are the backbone of *Matter Intelligence* reporting.

### 📝 Recap — what you shipped

- Data is split across tables (each fact stored once); a **`JOIN ... ON` key** re-connects them.
- **`INNER JOIN`** keeps only matches; **`LEFT JOIN`** keeps every left-table row (non-matches → `NULL`).
- **Table aliases** (`o`, `m`) and qualified columns (`o.price`) keep multi-table SQL readable and unambiguous.
- **Join + `GROUP BY`** produces cross-table reports — profit by region, billings by industry.
- **Artifact:** `profit_by_store()` — a join-then-group report with a real margin column.

### 🧠 Check your understanding

1. What's the difference between an `INNER JOIN` and a `LEFT JOIN`?
2. Why must you qualify `item` as `o.item` when both joined tables have an `item` column?
3. What happens if you write a `JOIN` with no `ON` condition?
4. Which join is the pandas `merge(..., how="left")`?

<details><summary>✅ Answers</summary>

1. `INNER` returns only rows that match in **both** tables; `LEFT` returns **all** left-table rows, filling the right side with `NULL` where there's no match.
2. The name is **ambiguous** — SQL doesn't know which table's `item` you mean, so you prefix it with the table (alias).
3. You get a **cross join**: every row of one table paired with every row of the other (usually a mistake and a huge result).
4. **`LEFT JOIN`.**
</details>

### ➡️ Next up — Week 6, Day 4: SQL **and** Python together

You've learned to `SELECT`, filter, group, and join — all in the warehouse. Tomorrow we make SQL and Python a **team**: let SQL do the heavy set-based work at scale, hand the result to **pandas** for shaping and a **matplotlib** chart, and tee it up for Claude. The "best of both" pipeline.

*Next lesson adds a quick `matplotlib` install (auto-handled) — nothing to do now.*

### 📖 Reference & glossary

| Term | Plain meaning | pandas twin |
|---|---|---|
| **`JOIN` / `INNER JOIN`** | combine tables, keep matches only | `merge(..., how="inner")` |
| **`LEFT JOIN`** | keep all left rows, `NULL` where no match | `merge(..., how="left")` |
| **`ON`** | the matching condition (the key) | `on="col"` |
| **Key** | shared column tables match on | the merge key |
| **Table alias** | short name for a table (`o`) | — |
| **`NULL`** | "no value" (shows as `None`/`NaN`) | `NaN` |
| **Cross join** | every row × every row (usually a bug) | `merge(..., how="cross")` |

**Docs:** Snowflake joins — https://docs.snowflake.com/en/sql-reference/constructs/join

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*